# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/siddanger7/flyrank-ml-internship-/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames my Week-1 research question and provisional lane. The framing lives in the markdown cells; the code cells show the real numbers that back it.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/siddanger7/flyrank-ml-internship-"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found. Ready.")

Working dir: C:\Users\sidda\AppData\Local\Temp\opencode\repo
Starter data found. Ready.


## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.**

I want to answer: *which visible pages under-capture clicks for their position tier, and should be reviewed first for a title/meta or content update?*

Why this lane:
- The starter data has the strongest direct signal for it — impressions, clicks, CTR, average position, position tier, sessions, and engagement columns are all present and measured, not derived.
- It produces a *ranked list* a human can act on (limited review capacity, thousands of pages), which is exactly the kind of decision-support problem this track is about.
- The comparison is honest by construction: CTR only makes sense *within* a position tier (page 1 always clicks better than page 9), and this lane makes that adjustment explicit.
- I already saw the anchor finding in Notebook 01: mean CTR collapses from ~0.35 on page 1 to ~0.05 deep in the results — so there is real, visible spread to work with.

I can confirm or change this lane until the end of Week 4; for now it is the best fit between the data I have, a decision someone actually makes, and a metric I can defend.

In [2]:
# How big is the surface this lane would rank? Let's measure it before committing.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df), "| Columns:", df.shape[1])

# Pages with real impressions AND a real position are the ones CTR can speak about.
has_pos = (df["impressions_90d"] > 0) & (df["avg_position"] > 0)
print("Pages with impressions_90d > 0 and avg_position > 0:", int(has_pos.sum()))
print("Share of the inventory this lane could rank: {:.0%}".format(has_pos.mean()))

Rows: 30000 | Columns: 44
Pages with impressions_90d > 0 and avg_position > 0: 28795
Share of the inventory this lane could rank: 96%


## 2. The question: decision, action, cost of a wrong call

**The decision I want to improve:** which visible pages a content editor reviews *first* for click-through-rate / title-meta improvement.

**Who acts on it:** a search content editor with limited weekly review capacity (a few dozen pages at most).

**The action they take:** rewrite the title/meta (or the on-page content) of the pages ranked highest, and skip the rest — so the output must be a ranked, explainable list, not a pile of numbers.

**What a wrong recommendation costs:**
- Recommending a page that already captures clicks: wasted editor hours and an unnecessary, risky rewrite.
- Missing a genuinely under-capturing page: recoverable clicks stay on the table.
- Because capacity is small, *precision at the top of the list* matters more than overall accuracy — this is why top-K metrics (Precision@20 / @50), not raw accuracy, are the right evaluation.

**Why data / ML helps at all:** a naive rule like “low CTR = bad page” is wrong, because CTR depends heavily on position. The real signal is messier: low CTR *relative to position tier and volume*, among many interacting columns, with low-volume noise to filter. A learned model can capture “weak for where it ranks, at scale” better than any hand-written threshold — but it must always be compared against a transparent baseline rule.

In [3]:
# The cost side, in numbers: how many pages are even 'visible enough to matter'?
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[df["impressions_90d"] >= 500]
print("Pages with impressions_90d >= 500 (enough exposure to matter):", len(visible))
print("Share of all pages: {:.0%}".format(len(visible) / len(df)))

# An editor with ~50 review slots per cycle faces this many candidates — the rank matters.
print("\nReview capacity is the scarce resource; a top-K ranked list is the deliverable.")

Pages with impressions_90d >= 500 (enough exposure to matter): 16726
Share of all pages: 56%

Review capacity is the scarce resource; a top-K ranked list is the deliverable.


## 3. Quick look at the data (2-3 real numbers)

Three live numbers, computed from the shipped starter CSV, that make this lane look worth the next 7 weeks:

1. **The CTR cliff by position tier** — median CTR is several times higher at the top of the results than deep in them. If we don't adjust for position, we will mislabel every deep page.
2. **How many visible pages sit below their own tier's median** — this is the opportunity surface a reviewer can act on.
3. **The actionable core** — high-demand, well-positioned pages that still under-capture. These are the highest-value candidates.

Column gotchas honored: `ctr` is a ×100 rate (0.76 = 0.76%), and `avg_position == 0` means “no data”, so I filter those rows out rather than treating rank zero as real.

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Keep pages with real impressions AND a real position (avg_position == 0 means no data).
vis = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

# 1) The CTR cliff by position tier.
tier = vis.groupby("position_tier")["ctr"].agg(["count", "median"])
print("Median CTR by position tier (rate, %):")
print(tier.round(3).to_string())

# 2) Visible pages that under-capture clicks vs their own tier's median.
vis["tier_median"] = vis["position_tier"].map(tier["median"])
under = vis[vis["ctr"] < 0.5 * vis["tier_median"]]
print(f"\nVisible pages (impressions>=100, has position): {len(vis)}")
print(f"Under-capturing (CTR < half the tier median):   {len(under)} ({len(under)/len(vis):.0%})")

# 3) The actionable core: high demand, good position, still weak CTR.
strong = vis[(vis["impressions_90d"] >= 500) & (vis["avg_position"] <= 10) & (vis["ctr"] < 0.5 * vis["tier_median"])]
print("\nHigh-demand, well-positioned pages still under-capturing:", len(strong))
print("These are the highest-value review candidates for this lane.")

Median CTR by position tier (rate, %):
               count  median
position_tier               
deep             879    0.00
page_1          8633    0.23
page_3_5        6058    0.06
striking        5903    0.15
top_3            533    0.19

Visible pages (impressions>=100, has position): 22006
Under-capturing (CTR < half the tier median):   7239 (33%)

High-demand, well-positioned pages still under-capturing: 1825
These are the highest-value review candidates for this lane.


## 4. Careful words: what I can and can't claim

**I can claim (observed / measured / directional / decision-support):**
- “In this starter sample, median CTR is ~0.23% on page 1 vs ~0.06% on page 3-5 and 0.00% in the deep tier — a large observed gap.”
- “In this starter sample, 33% of visible pages sit below half their position tier's median CTR — a candidate surface for review.”
- “This ranks pages for a human reviewer; it is decision support, not an automated decision.”

**I cannot claim:**
- That a rewrite *caused* a CTR or traffic recovery — that would need a controlled experiment or causal design this data does not provide.
- That low CTR always means a bad title or meta — the real cause could be intent mismatch, SERP features, or AI click-loss.
- Any Google algorithm factor, or that I “predict Google.”
- “No AI sessions” as proof AI can't understand a page — absence of sessions is not proof of absence of visibility.

**Rules I will keep:** never use `trend_direction` or `trend_pct` as features (they encode the label); never compare CTR across positions without tier adjustment; require minimum volume so noise doesn't drive recommendations; and keep every published claim worded as observed and directional.

In [5]:
# Leakage guardrails for this lane, checked in code so they are visible in the notebook.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
label_encoders = ["trend_direction", "trend_pct"]
print("Label-derived columns that must NEVER become features:", label_encoders)

# avg_position == 0 means 'no data', not rank zero — this lane filters it everywhere.
print("Rows with avg_position == 0 (no data, must filter):", int((df["avg_position"] == 0).sum()))

# CTR is a percent-as-number (0.76 = 0.76%), never multiplied blindly as a fraction.
print("CTR is stored as % rate — max value:", df["ctr"].max())

Label-derived columns that must NEVER become features: ['trend_direction', 'trend_pct']
Rows with avg_position == 0 (no data, must filter): 1205
CTR is stored as % rate — max value: 100.0


## Self-check

Before submitting, each line is confirmed:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.